# ELL 409 Project 2 — High-Dimensional Clustering

## Overview

This notebook implements a part-specific clustering pipeline for ELL 409 Course Project 2. The dataset contains high-dimensional biological measurements across four structurally distinct subproblems (Parts A–D), each requiring a different clustering strategy.

| Part | Challenge | Algorithm |
|------|-----------|-----------|
| A | Sparse marker signals embedded in high-dimensional noise | Gaussian Mixture Model (BIC-based K selection) |
| B | Rare cell-type discovery under severe cluster imbalance | UMAP → HDBSCAN |
| C | Hierarchical and continuously transitioning cell states | Spectral Clustering (5-seed stability sweep) |
| D | Spatial tissue-niche clustering with neighbourhood context | Ward agglomerative + spatial features (3D sweep) |

## Shared Preprocessing Backbone

All four parts go through the same five-step pipeline before any clustering algorithm is applied:

1. **Variance-based gene filtering** — keep only the top-N most variable genes per part; discard uninformative noise dimensions.
2. **Median imputation** — fill missing protein and morphology values with per-column training medians.
3. **Batch mean centering** — subtract per-batch column means to remove systematic inter-batch offsets.
4. **Robust scaling** — normalise each feature by IQR, making scaling resistant to outliers.
5. **PCA** — project to a compact low-dimensional embedding that concentrates cluster-separating variance.

Part-specific algorithms run on the resulting embedding. Hyperparameters for Parts B, C, and D are chosen automatically by data-driven criteria (stability sweeps + bootstrap ARI), not by hand.


## 0. Setup and Dependencies

In [ ]:
# Install UMAP and HDBSCAN if not already present (Colab-friendly).
# Comment this out if you've already installed them.
import importlib, subprocess, sys

for pkg, import_name in [("umap-learn", "umap"), ("hdbscan", "hdbscan")]:
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import (
    AgglomerativeClustering, DBSCAN, KMeans, SpectralClustering,
)
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    calinski_harabasz_score, davies_bouldin_score, silhouette_score,
    adjusted_rand_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from sklearn.preprocessing import RobustScaler

# Try to import UMAP and HDBSCAN. If unavailable, the pipeline falls back gracefully.
# On Colab you may need: !pip install -q umap-learn hdbscan
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("umap-learn not available. Run: !pip install -q umap-learn")

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except ImportError:
    HDBSCAN_AVAILABLE = False
    print("hdbscan not available. Run: !pip install -q hdbscan")

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/ELL409"

Mounted at /content/drive


In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

# Named column groups used throughout the notebook
GENES    = [f"g{i}"     for i in range(1, 501)]
PROTEINS = [f"p{i}"     for i in range(1, 41)]
MORPHS   = [f"morph{i}" for i in range(1, 31)]
QC       = ["qc_depth", "qc_noise", "dropout_rate"]
SPATIAL  = ["x_coord", "y_coord", "neighbor_density", "local_heterogeneity"]

# Per-part configuration dictionary.
# All values here are either static or serve as safe defaults that get
# overwritten by the hyperparameter sweeps before the pipeline runs.
PART_CONFIG = {
    # Part A: sparse marker clustering.
    # Diagonal GMM covariance chosen over full: at d=30, full covariance has
    # d*(d+1)/2 = 465 parameters per component vs. d=30 for diagonal, making
    # full covariance prone to ill-conditioning at cluster sizes of ~100-300.
    # k_grid: BIC is computed at each K and the minimum BIC value is selected.
    "A": {"top_genes": 180, "pca_dim": 30, "n_clusters": 10,
          "covariance_type": "diag",
          "k_grid": [7, 8, 9, 10, 11, 12, 13]},
    # Part B: rare cell-type discovery.
    # umap_min_dist=0.0 collapses same-neighbourhood points as tightly as
    # possible, sharpening cluster boundaries for HDBSCAN density estimation.
    # noise_split_threshold=0.05: if noise fraction < 5%, all noise points
    # are assigned to one extra cluster; otherwise KMeans splits them.
    "B": {"top_genes": 220, "pca_dim": 35,
          "dbscan_eps": 7.0,
          "min_samples": 10,
          "umap_n_neighbors": 15,
          "umap_min_dist": 0.0,
          "umap_dim": 10,
          "min_cluster_size": 8,
          "hdbscan_min_samples": 5,
          "noise_split_threshold": 0.05,
          "noise_kmeans_k": 2,
          },
    # Part C: hierarchical and continuous cell states.
    # n_clusters and n_neighbors are overwritten by the 5-seed stability sweep.
    # Stability threshold >= 0.5 (mean pairwise ARI) is the selection criterion;
    # silhouette is only a tie-break — it does NOT reliably predict ARI here.
    "C": {"top_genes": 240, "pca_dim": 40,
          "n_clusters": 16,
          "n_neighbors": 20},
    # Part D: spatial tissue-niche clustering.
    # n_clusters, smooth_weight_mean, smooth_weight_std, spatial_coord_weight
    # are set by the 3D sweep. Defaults are safe fallbacks if sweep is disabled.
    "D": {"top_genes": 180, "pca_dim": 30,
          # These four are set by the 3D sweep. Defaults are safe fallbacks
          # if the sweep is disabled.
          "n_clusters": 12,
          "smooth_weight_mean": 0.3,
          "smooth_weight_std":  0.15,        # locked to 0.5 * mean weight
          "spatial_coord_weight": 0.5,
          # Static config (not swept)
          "spatial_neighbors": 8,
          # Sweep grids for the 3D hyperparameter search
          "k_grid":              [8, 10, 12, 14, 16],
          "mean_weight_grid":    [0.1, 0.2, 0.3, 0.5],
          "spatial_weight_grid": [0.25, 0.5, 1.0],
          # std weight is locked to 0.5 * mean weight to preserve the mean-to-std ratio
          },
}

# Feature flags — toggle individual pipeline components on or off
USE_MORE_EM_RESTARTS    = True
USE_BEST_OF_N_SPECTRAL  = True
USE_LOG1P_GENES         = False
USE_AUTO_DBSCAN_EPS     = False

# Additional feature flags
USE_BIC_FOR_PART_A_K      = True
USE_HDBSCAN_PART_B        = True and UMAP_AVAILABLE and HDBSCAN_AVAILABLE
USE_5_SEED_PART_C         = True
USE_BLOCK_STANDARDIZATION = True

# Part D 3D sweep flag
USE_PART_D_3D_SWEEP       = True

print("train shape:", train.shape)
print("test shape:",  test.shape)
print()
print("v9 flags:")
print(f"  USE_BIC_FOR_PART_A_K      = {USE_BIC_FOR_PART_A_K}")
print(f"  USE_HDBSCAN_PART_B        = {USE_HDBSCAN_PART_B}")
print(f"  USE_5_SEED_PART_C         = {USE_5_SEED_PART_C}")
print(f"  USE_BLOCK_STANDARDIZATION = {USE_BLOCK_STANDARDIZATION}")
print("v12 flag:")
print(f"  USE_PART_D_3D_SWEEP       = {USE_PART_D_3D_SWEEP}")


In [ ]:
# log1p requires non-negative inputs. Check the minimum gene value first.
# If any gene value is negative, log1p is skipped entirely.
_gene_min = train[GENES].min().min()
LOG1P_APPLICABLE = bool(_gene_min >= 0) and USE_LOG1P_GENES
APPLY_LOG1P = LOG1P_APPLICABLE
print(f"min gene value: {_gene_min:.4f}; APPLY_LOG1P = {APPLY_LOG1P}")

min gene value: 0.0000; APPLY_LOG1P = False


## 1. Helper Functions

In [ ]:
def maybe_log1p_genes(df, cols):
    # Applies log1p to gene columns only if APPLY_LOG1P is True.
    # Because the minimum gene value is negative (-6.19), APPLY_LOG1P is False
    # throughout and this function is effectively a no-op.
    out = df[cols].astype(float).copy()
    if APPLY_LOG1P:
        gene_cols_in = [c for c in cols if c.startswith("g") and c[1:].isdigit()]
        if gene_cols_in:
            vals = out[gene_cols_in].fillna(0).to_numpy()
            out[gene_cols_in] = np.log1p(np.clip(vals, a_min=0, a_max=None))
    return out


def top_variable_columns(reference_df, cols, top_n):
    # Returns the names of the top_n columns with highest variance in reference_df.
    # Called per-part on training rows so gene rankings reflect within-part variation.
    vs_input = maybe_log1p_genes(reference_df, cols)
    variances = vs_input.var(numeric_only=True).sort_values(ascending=False)
    return variances.head(top_n).index.tolist()


def fit_numeric_embedding(df, cols, pca_dim, do_batch_center=True):
    # Shared 5-step preprocessing backbone used by all four parts:
    # 1. (log1p — disabled) 2. Median imputation  3. Per-batch mean centering
    # 4. RobustScaler (IQR-based, resistant to outliers)  5. PCA
    X = maybe_log1p_genes(df, cols)
    X_arr = SimpleImputer(strategy="median").fit_transform(X)
    X_arr = pd.DataFrame(X_arr, columns=cols, index=df.index)

    if do_batch_center and "batch_id" in df.columns:
        # Subtract per-batch column means to remove systematic batch offsets.
        # Without this, PCA components reflect batch identity rather than biology.
        X_arr["__b"] = df["batch_id"].values
        X_arr[cols] = X_arr.groupby("__b")[cols].transform(lambda x: x - x.mean())
        X_arr = X_arr[cols]

    X_arr = RobustScaler().fit_transform(X_arr.to_numpy())
    pca_dim = min(pca_dim, X_arr.shape[0] - 1, X_arr.shape[1] - 1)
    pca = PCA(n_components=pca_dim, random_state=RANDOM_STATE)
    Z = pca.fit_transform(X_arr)
    return Z, {"pca": pca, "columns": cols}


def clustering_metrics(embedding, labels):
    # Computes the three internal metrics reported in Section 7 of the report:
    # silhouette (higher = better), Davies-Bouldin (lower = better),
    # and Calinski-Harabasz (higher = better).
    # Silhouette is sampled on up to 2000 points to control runtime.
    labels = np.asarray(labels)
    n_clusters = len(np.unique(labels))
    if n_clusters < 2:
        return {"n_clusters": n_clusters, "silhouette": np.nan,
                "davies_bouldin": np.nan, "calinski_harabasz": np.nan}
    sample_size = min(2000, len(embedding))
    return {
        "n_clusters": n_clusters,
        "silhouette":     silhouette_score(embedding, labels,
                                            sample_size=sample_size,
                                            random_state=RANDOM_STATE),
        "davies_bouldin": davies_bouldin_score(embedding, labels),
        "calinski_harabasz": calinski_harabasz_score(embedding, labels),
    }


def relabel_by_cluster_size(labels):
    # Renumbers cluster IDs so cluster 0 is the largest, 1 is second largest, etc.
    # Makes outputs consistent and comparable across runs.
    counts = pd.Series(labels).value_counts().sort_values(ascending=False)
    mapping = {old: new for new, old in enumerate(counts.index.tolist())}
    return np.array([mapping[label] for label in labels], dtype=int)


def summarize_cluster_sizes(labels):
    counts = pd.Series(labels).value_counts().sort_index()
    return pd.DataFrame({"cluster_id": counts.index, "size": counts.values})


# ── Spatial and stability helpers ─────────────────────────────────────────────

def standardize_block(X):
    """Centre a feature block and rescale to unit RMS.

    After concatenation, a weight multiplier on the block then controls its
    contribution to Euclidean distance in a scale-invariant way. Without this,
    whichever block has the largest natural scale would dominate regardless of
    the chosen weight.
    """
    X = np.asarray(X, dtype=float)
    X = X - X.mean(axis=0, keepdims=True)
    rms = float(np.sqrt((X ** 2).mean()))
    if rms > 1e-12:
        X = X / rms
    return X


def n_seed_spectral_stability(Z, n_clusters, n_neighbors, n_seeds=5):
    """Run SpectralClustering with n_seeds different RNG seeds and return
    (mean pairwise ARI across all runs, list of label arrays).

    Mean pairwise ARI measures how consistently the algorithm finds the same
    partition. High stability (ARI >= 0.5) indicates genuine structure rather
    than random variation. Used by the Part C sweep to select (k, n_neighbors).
    """
    runs = []
    for seed in range(42, 42 + n_seeds):
        try:
            sc = SpectralClustering(
                n_clusters=n_clusters,
                affinity="nearest_neighbors",
                n_neighbors=n_neighbors,
                assign_labels="kmeans",
                random_state=seed,
            )
            labels = sc.fit_predict(Z)
            if len(np.unique(labels)) >= 2:
                runs.append(labels)
        except Exception:
            continue
    if len(runs) < 2:
        return np.nan, runs
    aris = []
    for i in range(len(runs)):
        for j in range(i + 1, len(runs)):
            aris.append(adjusted_rand_score(runs[i], runs[j]))
    return float(np.mean(aris)), runs


In [ ]:
# Build per-part feature lists from the training data.
# Gene selection is computed on each part's rows separately so that the
# top-N genes are the most variable *within that part*, not globally.
train_feature_choices = {}
for part, cfg in PART_CONFIG.items():
    df_p = train[train["part"] == part]
    train_feature_choices[part] = {
        "genes":    top_variable_columns(df_p, GENES, cfg["top_genes"]),
        "proteins": PROTEINS.copy(),
        "morphs":   MORPHS.copy(),
    }

# For single-cell parts (A, B, C): concatenate genes + proteins + morphs + QC
for p in ["A", "B", "C"]:
    fc = train_feature_choices[p]
    fc["base_cols"] = fc["genes"] + fc["proteins"] + fc["morphs"] + QC

# For Part D: store as "molecular_cols" — spatial features are added separately
train_feature_choices["D"]["molecular_cols"] = (
    train_feature_choices["D"]["genes"]
    + train_feature_choices["D"]["proteins"]
    + train_feature_choices["D"]["morphs"]
    + QC
)

## 2. Hyperparameter Sweeps (Parts B and C)

Parts B and C require searching over algorithm-specific parameters on the training data. The best configuration is written back into `PART_CONFIG` and then used by the pipeline functions.

- **Part B**: sweeps `min_cluster_size` and `min_samples` for HDBSCAN on the UMAP embedding. Selects the configuration that yields a valid K (6–20 total clusters) with the lowest noise fraction.
- **Part C**: sweeps `n_clusters` and `n_neighbors` for SpectralClustering. Selects the most stable configuration as measured by mean pairwise ARI across 5 random seeds (threshold ≥ 0.5).

In [ ]:
# --- Part B: HDBSCAN hyperparameter sweep ---
# Sweep min_cluster_size and min_samples and pick the config that yields a K
# in the allowed range (6..20) with the fewest noise points and the smallest
# minimum cluster size (i.e., genuinely rare clusters are preserved).
#
# If HDBSCAN/UMAP not available, fall back to a DBSCAN eps sweep.

df_b = train[train["part"] == "B"].reset_index(drop=True)
Z_b, _ = fit_numeric_embedding(df_b,
    train_feature_choices["B"]["base_cols"],
    pca_dim=PART_CONFIG["B"]["pca_dim"])

if USE_HDBSCAN_PART_B:
    # Build a single UMAP embedding once, sweep HDBSCAN params on it.
    # min_dist=0.0 collapses same-neighbourhood points tightly, sharpening
    # the density signal that HDBSCAN uses to detect rare clusters.
    reducer = umap.UMAP(
        n_neighbors=PART_CONFIG["B"]["umap_n_neighbors"],
        min_dist=PART_CONFIG["B"]["umap_min_dist"],
        n_components=PART_CONFIG["B"]["umap_dim"],
        random_state=RANDOM_STATE,
    )
    Z_b_umap = reducer.fit_transform(Z_b)

    sweep_b = []
    for mcs in [5, 6, 8, 10, 12, 15]:
        for ms in [3, 5, 8]:
            try:
                hdb = hdbscan.HDBSCAN(
                    min_cluster_size=mcs,
                    min_samples=ms,
                    cluster_selection_method="eom",
                ).fit(Z_b_umap)
                labels = hdb.labels_
            except Exception:
                continue
            kept = sorted([l for l in np.unique(labels) if l >= 0])
            n_dense = len(kept)
            noise_pct = 100.0 * (labels == -1).mean()
            min_size = (
                min(int((labels == c).sum()) for c in kept) if kept else 0
            )
            sweep_b.append({
                "min_cluster_size": mcs, "min_samples": ms,
                "n_dense": n_dense, "noise_pct": noise_pct,
                "min_size": min_size,
            })

    df_sweep = pd.DataFrame(sweep_b)
    print("Part B HDBSCAN sweep:")
    print(df_sweep.to_string(index=False))

    # Selection: K in [6, 20] AFTER adding 1 cluster for noise (so dense in [5, 19]).
    # Prefer configs with low noise, then low min_size (keep rare clusters).
    valid = df_sweep[(df_sweep["n_dense"] >= 5) & (df_sweep["n_dense"] <= 19)]
    if len(valid) > 0:
        valid = valid.sort_values(["noise_pct", "min_size"], ascending=[True, True])
        chosen = valid.iloc[0]
        PART_CONFIG["B"]["min_cluster_size"]    = int(chosen["min_cluster_size"])
        PART_CONFIG["B"]["hdbscan_min_samples"] = int(chosen["min_samples"])
        print(f"\nChose: min_cluster_size={PART_CONFIG['B']['min_cluster_size']}, "
              f"min_samples={PART_CONFIG['B']['hdbscan_min_samples']}, "
              f"K_dense={int(chosen['n_dense'])}, noise={chosen['noise_pct']:.1f}%")
    else:
        print("\nNo valid HDBSCAN config in K range. Falling back to v8 DBSCAN at runtime.")
        USE_HDBSCAN_PART_B = False  # forces fallback in pipeline
else:
    print("USE_HDBSCAN_PART_B is False; running v8 DBSCAN eps sweep instead.")

# Fallback: DBSCAN eps sweep (used if HDBSCAN is unavailable or finds no valid config)
if not USE_HDBSCAN_PART_B:
    sweep_b_v8 = []
    for eps in [5.0, 6.0, 6.5, 7.0, 7.5, 8.0, 9.0]:
        dense = DBSCAN(eps=eps, min_samples=PART_CONFIG["B"]["min_samples"]).fit_predict(Z_b)
        kept = sorted([l for l in np.unique(dense) if l >= 0])
        n_dense = len(kept)
        noise_pct = 100.0 * (dense == -1).mean()
        sil = float("nan")
        if n_dense >= 2:
            non_noise = dense != -1
            if non_noise.sum() >= 2 and len(np.unique(dense[non_noise])) >= 2:
                sample = min(2000, non_noise.sum())
                sil = silhouette_score(Z_b[non_noise], dense[non_noise],
                                        sample_size=sample, random_state=RANDOM_STATE)
        sweep_b_v8.append((eps, n_dense, noise_pct, sil))
    valid_b = [(eps, k, n, s) for (eps, k, n, s) in sweep_b_v8
               if 6 <= k <= 20 and not np.isnan(s)]
    if valid_b:
        valid_b.sort(key=lambda t: (t[2], -t[3]))
        PART_CONFIG["B"]["dbscan_eps"] = valid_b[0][0]
    print(f"Part B DBSCAN fallback eps -> {PART_CONFIG['B']['dbscan_eps']}")


In [ ]:
# --- Part C: SpectralClustering grid sweep (5-seed stability) ---
# Sweep k (number of clusters) and nn (k-NN graph neighbours).
# For each config, n_seed_spectral_stability() runs 5 seeds and returns the
# mean pairwise ARI across all C(5,2)=10 pairs as the stability score.
# Selection rule: stability >= 0.5 required; tie-break by silhouette.
# Note: silhouette is NOT the primary criterion here — high silhouette does not
# imply high ARI for manifold-structured data (see report Section 8.2).
df_c = train[train["part"] == "C"].reset_index(drop=True)
Z_c, _ = fit_numeric_embedding(df_c,
    train_feature_choices["C"]["base_cols"],
    pca_dim=PART_CONFIG["C"]["pca_dim"])

n_seeds = 5 if USE_5_SEED_PART_C else 2

results_c = []
for k in [12, 14, 16, 18]:
    for nn in [15, 20, 30]:
        stab, runs = n_seed_spectral_stability(Z_c, k, nn, n_seeds=n_seeds)
        if np.isnan(stab) or len(runs) == 0:
            continue
        # Use the first valid run for silhouette
        sample = min(2000, len(Z_c))
        sil = silhouette_score(Z_c, runs[0], sample_size=sample,
                                random_state=RANDOM_STATE)
        results_c.append((k, nn, stab, sil))

print(f"Part C sweep (n_seeds={n_seeds}, mean pairwise ARI):")
print(f"{'k':<5}{'nn':<5}{'stab':<8}{'sil':<8}")
for k, nn, stab, sil in sorted(results_c, key=lambda t: (-t[2], -t[3])):
    print(f"{k:<5}{nn:<5}{stab:<8.3f}{sil:<8.3f}")

stable_c = [(k, nn, stab, sil) for (k, nn, stab, sil) in results_c if stab >= 0.5]
if stable_c:
    stable_c.sort(key=lambda t: (-t[2], -t[3]))
    PART_CONFIG["C"]["n_clusters"]  = stable_c[0][0]
    PART_CONFIG["C"]["n_neighbors"] = stable_c[0][1]
print(f"\nPart C -> n_clusters={PART_CONFIG['C']['n_clusters']}, "
      f"n_neighbors={PART_CONFIG['C']['n_neighbors']}")


## 3. Part D: 3D Hyperparameter Sweep

Part D uses three jointly tuned parameters: number of clusters (K), neighbourhood mean weight (α), and spatial coordinate weight (β). Sweeping them individually would miss interactions. The search grid below covers all combinations; `smooth_weight_std` is locked to `0.5 × smooth_weight_mean` to keep the sweep three-dimensional.

| Parameter | Values | Count |
|-----------|--------|-------|
| `n_clusters` | 8, 10, 12, 14, 16 | 5 |
| `smooth_weight_mean` | 0.1, 0.2, 0.3, 0.5 | 4 |
| `spatial_coord_weight` | 0.25, 0.5, 1.0 | 3 |

Total: 60 configurations × 2 bootstrap subsets = 120 Ward fits.

For each configuration the pipeline is run on two independent 90% subsamples of the training data. Bootstrap stability (ARI on the overlap) is used for selection — the most stable configuration wins, with silhouette as a tie-break.

In [ ]:
df_d = train[train["part"] == "D"].reset_index(drop=True)

def part_d_pipeline(df_subset, n_clusters, mean_w, std_w, coord_w):
    """Run the Part D clustering pipeline with explicit weight and K parameters.

    Builds a molecular PCA embedding, computes spatial neighbourhood statistics
    (mean and std of neighbours' embeddings), applies block standardisation so
    that each weight multiplier is scale-invariant, then runs Ward agglomerative
    clustering on the concatenated embedding.

    Used both during the hyperparameter sweep and by cluster_part_d().
    """
    cols = train_feature_choices["D"]["molecular_cols"]
    Z, _ = fit_numeric_embedding(df_subset, cols, pca_dim=PART_CONFIG["D"]["pca_dim"])
    coords = df_subset[["x_coord", "y_coord"]].to_numpy()
    n_neigh = min(PART_CONFIG["D"]["spatial_neighbors"], len(df_subset) - 1)
    # Build binary k-NN spatial graph: A[i,j]=1 iff spot j is among spot i's
    # 8 nearest spatial neighbours.
    graph = kneighbors_graph(coords, n_neighbors=n_neigh,
                              mode="connectivity", include_self=False)
    row_sum = np.asarray(graph.sum(axis=1)).ravel()
    denom = np.maximum(row_sum[:, None], 1)
    # Neighbourhood mean: average PCA embedding of each spot's 8 spatial neighbours.
    # Captures "what does the surrounding tissue look like molecularly?"
    neighbor_mean = (graph @ Z) / denom
    # Neighbourhood std: spread of neighbours' embeddings.
    # High std indicates a spot sits at a niche boundary.
    neighbor_sq   = (graph @ (Z ** 2)) / denom
    neighbor_std  = np.sqrt(np.maximum(neighbor_sq - neighbor_mean ** 2, 0.0))
    spatial_block = df_subset[SPATIAL].to_numpy()

    if USE_BLOCK_STANDARDIZATION:
        # Block standardisation makes each block unit-RMS so that weight
        # multipliers are scale-invariant across blocks (see report Section 4.4.3).
        Z_blk  = standardize_block(Z)
        nm_blk = standardize_block(neighbor_mean)
        ns_blk = standardize_block(neighbor_std)
        sp_blk = standardize_block(spatial_block)
    else:
        Z_blk, nm_blk, ns_blk, sp_blk = Z, neighbor_mean, neighbor_std, spatial_block
        coord_w = 1.0  # v8 behavior

    # Final augmented embedding: Z | alpha*NM | (alpha/2)*NS | beta*SP
    embedding = np.hstack([
        Z_blk,
        mean_w  * nm_blk,
        std_w   * ns_blk,
        coord_w * sp_blk,
    ])
    # Ward linkage is deterministic (no random init), ideal for stability sweeps.
    labels = AgglomerativeClustering(n_clusters=n_clusters,
                                      linkage="ward").fit_predict(embedding)
    return labels, embedding


# Bootstrap stability setup: draw two independent 90% subsamples of Part D
# training rows. The overlap S_a ∩ S_b is where we compare labels from both runs.
# ARI on the overlap measures how consistently a configuration clusters the data.
# Set up bootstrap subsets once, reuse across all 60 configs
rng = np.random.RandomState(RANDOM_STATE)
n_total = len(df_d)
idx_a = rng.choice(n_total, size=int(0.9 * n_total), replace=False)
idx_b = rng.choice(n_total, size=int(0.9 * n_total), replace=False)
overlap = np.intersect1d(idx_a, idx_b)
df_subA = df_d.iloc[idx_a].reset_index(drop=True)
df_subB = df_d.iloc[idx_b].reset_index(drop=True)
pos_a = {orig: i for i, orig in enumerate(idx_a)}
pos_b = {orig: i for i, orig in enumerate(idx_b)}

if USE_PART_D_3D_SWEEP:
    # 3D sweep: 5 K values x 4 alpha values x 3 beta values = 60 configs.
    # std_w is locked to 0.5*mean_w to preserve the mean-to-std ratio and
    # keep the sweep three-dimensional. Total: 60 configs x 2 subsets = 120 fits.
    K_GRID    = PART_CONFIG["D"]["k_grid"]
    MEAN_GRID = PART_CONFIG["D"]["mean_weight_grid"]
    COORD_GRID = PART_CONFIG["D"]["spatial_weight_grid"]
    n_combos = len(K_GRID) * len(MEAN_GRID) * len(COORD_GRID)
    print(f"Part D 3D sweep: {len(K_GRID)} K × {len(MEAN_GRID)} mean_w × "
          f"{len(COORD_GRID)} coord_w = {n_combos} configs")
    print(f"{'k':<4}{'mean_w':<8}{'coord_w':<10}{'stab':<8}{'sil':<8}{'note'}")
    print("-" * 50)

    results_d = []
    for k in K_GRID:
        for mw in MEAN_GRID:
            sw = 0.5 * mw  # std weight locked to 0.5 * mean weight
            for cw in COORD_GRID:
                try:
                    labels_a, emb_a = part_d_pipeline(df_subA, k, mw, sw, cw)
                    labels_b, _     = part_d_pipeline(df_subB, k, mw, sw, cw)
                except Exception as e:
                    print(f"{k:<4}{mw:<8}{cw:<10} -- skipped ({type(e).__name__})")
                    continue
                # Bootstrap stability: ARI on the overlap between the two subsamples.
                # Configurations with stab >= 0.5 are considered stable.
                la_ov = np.array([labels_a[pos_a[o]] for o in overlap])
                lb_ov = np.array([labels_b[pos_b[o]] for o in overlap])
                stab = adjusted_rand_score(la_ov, lb_ov)
                sample = min(2000, len(emb_a))
                sil = silhouette_score(emb_a, labels_a, sample_size=sample,
                                        random_state=RANDOM_STATE) if len(np.unique(labels_a)) >= 2 else float("nan")
                note = "unstable" if stab < 0.5 else ""
                results_d.append((k, mw, cw, stab, sil, note))

    # Show top 10 by stability then silhouette
    print("\nTop 10 configs by (stab desc, sil desc):")
    print(f"{'k':<4}{'mean_w':<8}{'coord_w':<10}{'stab':<8}{'sil':<8}")
    for k, mw, cw, stab, sil, note in sorted(results_d,
                                               key=lambda t: (-t[3], -t[4]))[:10]:
        print(f"{k:<4}{mw:<8}{cw:<10}{stab:<8.3f}{sil:<8.3f}")
else:
    # Fallback: 1D sweep over K with weights fixed to current PART_CONFIG values
    GRID_K_D = [10, 12, 14, 16, 18, 20]
    print(f"{'k':<5} {'stability':<11} {'silhouette':<12} {'note'}")
    print("-" * 45)
    results_d = []
    mw_fixed = PART_CONFIG["D"]["smooth_weight_mean"]
    sw_fixed = PART_CONFIG["D"]["smooth_weight_std"]
    cw_fixed = PART_CONFIG["D"]["spatial_coord_weight"]
    for k in GRID_K_D:
        try:
            labels_a, emb_a = part_d_pipeline(df_subA, k, mw_fixed, sw_fixed, cw_fixed)
            labels_b, _     = part_d_pipeline(df_subB, k, mw_fixed, sw_fixed, cw_fixed)
        except Exception as e:
            print(f"{k:<5} -- skipped ({type(e).__name__})")
            continue
        la_ov = np.array([labels_a[pos_a[o]] for o in overlap])
        lb_ov = np.array([labels_b[pos_b[o]] for o in overlap])
        stab = adjusted_rand_score(la_ov, lb_ov)
        sample = min(2000, len(emb_a))
        sil = silhouette_score(emb_a, labels_a, sample_size=sample,
                                random_state=RANDOM_STATE)
        note = "unstable - skip" if stab < 0.5 else ""
        # Pad to 6 fields so the auto-pick cell can handle either format
        results_d.append((k, mw_fixed, cw_fixed, stab, sil, note))
        print(f"{k:<5} {stab:<11.3f} {sil:<12.3f} {note}")


In [ ]:
# Pick the (k, mean_w, coord_w) combo with highest stability, tie-break by silhouette.
stable_d = [(k, mw, cw, stab, sil) for (k, mw, cw, stab, sil, note) in results_d
            if stab >= 0.5 and not np.isnan(stab)]

if stable_d:
    stable_d.sort(key=lambda t: (-t[3], -t[4]))
    best = stable_d[0]
    PART_CONFIG["D"]["n_clusters"]            = best[0]
    PART_CONFIG["D"]["smooth_weight_mean"]    = best[1]
    PART_CONFIG["D"]["smooth_weight_std"]     = 0.5 * best[1]
    PART_CONFIG["D"]["spatial_coord_weight"]  = best[2]
    print(f"\nPart D auto-selected:")
    print(f"  n_clusters            = {PART_CONFIG['D']['n_clusters']}")
    print(f"  smooth_weight_mean    = {PART_CONFIG['D']['smooth_weight_mean']}")
    print(f"  smooth_weight_std     = {PART_CONFIG['D']['smooth_weight_std']}")
    print(f"  spatial_coord_weight  = {PART_CONFIG['D']['spatial_coord_weight']}")
    print(f"  stability = {best[3]:.3f}, silhouette = {best[4]:.3f}")

    v9_in_grid = any(t[0] == 10 and t[1] == 0.3 and t[2] == 0.5 for t in stable_d)
    if v9_in_grid:
        v9_row = [t for t in stable_d if t[0] == 10 and t[1] == 0.3 and t[2] == 0.5][0]
        print(f"\nFor reference, v9 config (K=10, mean=0.3, coord=0.5):")
        print(f"  stability = {v9_row[3]:.3f}, silhouette = {v9_row[4]:.3f}")
        delta_stab = best[3] - v9_row[3]
        delta_sil  = best[4] - v9_row[4]
        print(f"  v12 - v9: stab {delta_stab:+.3f}, sil {delta_sil:+.3f}")
        if delta_stab < 0.005 and delta_sil < 0.005:
            print(f"  (v12 chosen config is essentially equivalent to v9)")
else:
    print("\nNo stable config found. Falling back to v9 defaults: K=10, "
          "mean=0.3, std=0.15, coord=0.5")
    PART_CONFIG["D"]["n_clusters"]           = 10
    PART_CONFIG["D"]["smooth_weight_mean"]   = 0.3
    PART_CONFIG["D"]["smooth_weight_std"]    = 0.15
    PART_CONFIG["D"]["spatial_coord_weight"] = 0.5


In [ ]:
# Manual override slot. Useful if you eyeball the table and disagree with the auto-pick
# (e.g., a slightly less stable k has much better silhouette).

# PART_CONFIG["D"]["n_clusters"] = 12   # uncomment to override

print(f"Final Part D: n_clusters = {PART_CONFIG['D']['n_clusters']}")

Final Part D: n_clusters = 10


## 4. Part-Specific Clustering Pipelines

In [ ]:
def cluster_part_a(df):
    """Cluster Part A using a Gaussian Mixture Model.

    Runs a BIC sweep over k_grid to select the number of components, then
    fits the final GMM with multiple random restarts for robustness.
    BIC(K) = -2*log_likelihood + p_K*ln(n), p_K = K*(2d+1)-1.
    The penalty accounts for K mean vectors, K diagonal variance vectors,
    and K-1 free mixing weights. Minimum BIC selects K automatically.
    """
    cols = train_feature_choices["A"]["base_cols"]
    Z, _ = fit_numeric_embedding(df, cols, pca_dim=PART_CONFIG["A"]["pca_dim"])

    if USE_BIC_FOR_PART_A_K:
        bic_results = []
        for k in PART_CONFIG["A"]["k_grid"]:
            try:
                gm = GaussianMixture(
                    n_components=k,
                    covariance_type=PART_CONFIG["A"]["covariance_type"],
                    reg_covar=1e-4, n_init=3, max_iter=200,
                    random_state=RANDOM_STATE,
                ).fit(Z)
                bic_results.append((k, gm.bic(Z)))
            except Exception:
                continue
        if bic_results:
            bic_results.sort(key=lambda t: t[1])
            chosen_k = bic_results[0][0]
        else:
            chosen_k = PART_CONFIG["A"]["n_clusters"]
    else:
        chosen_k = PART_CONFIG["A"]["n_clusters"]

    # Re-fit with 10 restarts at the chosen K to avoid local optima.
    n_init = 10 if USE_MORE_EM_RESTARTS else 1
    model = GaussianMixture(
        n_components=chosen_k,
        covariance_type=PART_CONFIG["A"]["covariance_type"],
        reg_covar=1e-4, n_init=n_init, max_iter=200,
        random_state=RANDOM_STATE,
    )
    labels = model.fit_predict(Z)
    return relabel_by_cluster_size(labels), Z


def cluster_part_b(df):
    """Cluster Part B using UMAP + HDBSCAN.

    Compresses the PCA embedding with UMAP (sharpens local density), then
    applies HDBSCAN to detect clusters at variable density levels. Noise
    points are either grouped into one extra cluster or split with KMeans,
    depending on their count. Falls back to DBSCAN + centroid merge if
    HDBSCAN is unavailable or produces a degenerate result.
    """
    cols = train_feature_choices["B"]["base_cols"]
    Z, _ = fit_numeric_embedding(df, cols, pca_dim=PART_CONFIG["B"]["pca_dim"])

    if USE_HDBSCAN_PART_B:
        try:
            reducer = umap.UMAP(
                n_neighbors=PART_CONFIG["B"]["umap_n_neighbors"],
                min_dist=PART_CONFIG["B"]["umap_min_dist"],
                n_components=PART_CONFIG["B"]["umap_dim"],
                random_state=RANDOM_STATE,
            )
            Z_umap = reducer.fit_transform(Z)

            hdb = hdbscan.HDBSCAN(
                min_cluster_size=PART_CONFIG["B"]["min_cluster_size"],
                min_samples=PART_CONFIG["B"]["hdbscan_min_samples"],
                cluster_selection_method="eom",
            ).fit(Z_umap)
            labels = hdb.labels_
            kept = sorted([l for l in np.unique(labels) if l >= 0])

            if len(kept) >= 2:
                final = np.full(len(df), -1, dtype=int)
                for new, old in enumerate(kept):
                    final[labels == old] = new

                noise_idx = np.where(labels == -1)[0]
                if len(noise_idx) > 0:
                    noise_frac = len(noise_idx) / len(df)
                    # Adaptive noise handling (see report Section 4.2.4):
                    # small noise (<5% or <2*mcs) -> one extra cluster;
                    # large noise -> KMeans split to expose sub-structure.
                    if noise_frac < PART_CONFIG["B"]["noise_split_threshold"] \
                       or len(noise_idx) < 2 * PART_CONFIG["B"]["min_cluster_size"]:
                        final[noise_idx] = len(kept)
                    else:
                        n_noise_clusters = min(
                            PART_CONFIG["B"]["noise_kmeans_k"],
                            max(1, len(noise_idx) // PART_CONFIG["B"]["min_cluster_size"]),
                        )
                        if n_noise_clusters <= 1:
                            final[noise_idx] = len(kept)
                        else:
                            noise_lab = KMeans(
                                n_clusters=n_noise_clusters,
                                n_init=20, random_state=RANDOM_STATE,
                            ).fit_predict(Z_umap[noise_idx])
                            final[noise_idx] = len(kept) + noise_lab

                K_total = len(np.unique(final))
                if 6 <= K_total <= 20:
                    return relabel_by_cluster_size(final), Z
                # else: out of range, fall through to DBSCAN fallback
        except Exception as e:
            print(f"HDBSCAN+UMAP failed in cluster_part_b ({type(e).__name__}: {e}), "
                  f"falling back to v8 DBSCAN+merge.")

    # ---- Fallback: DBSCAN with centroid merge ----
    eps = PART_CONFIG["B"]["dbscan_eps"]
    dense = DBSCAN(eps=eps, min_samples=PART_CONFIG["B"]["min_samples"]).fit_predict(Z)
    kept = sorted([l for l in np.unique(dense) if l >= 0])

    if len(kept) == 0:
        labels = KMeans(n_clusters=8, n_init=20,
                        random_state=RANDOM_STATE).fit_predict(Z)
        return relabel_by_cluster_size(labels), Z

    final = np.full(len(df), -1, dtype=int)
    for new, old in enumerate(kept):
        final[dense == old] = new

    noise_idx = np.where(dense == -1)[0]
    if len(noise_idx) > 0:
        centroids = np.vstack([Z[final == c].mean(axis=0) for c in range(len(kept))])
        d = np.linalg.norm(Z[noise_idx][:, None, :] - centroids[None, :, :], axis=2)
        final[noise_idx] = d.argmin(axis=1)

    return relabel_by_cluster_size(final), Z


def cluster_part_c(df):
    """Cluster Part C using Spectral Clustering on a k-NN affinity graph.

    Runs n_runs independent seeds and returns the labelling with the highest
    silhouette score, reducing variance from the random KMeans step inside
    SpectralClustering. Hyperparameters (n_clusters, n_neighbors) are set
    by the stability sweep in the cell above.
    Spectral Clustering builds the normalised graph Laplacian L = I - D^{-1/2} A D^{-1/2}
    and clusters on its K smallest eigenvectors, capturing geodesic-like structure
    that flat distance-based methods (KMeans, GMM) cannot recover.
    """
    cols = train_feature_choices["C"]["base_cols"]
    Z, _ = fit_numeric_embedding(df, cols, pca_dim=PART_CONFIG["C"]["pca_dim"])
    n_runs = 3 if USE_BEST_OF_N_SPECTRAL else 1
    best_labels, best_score = None, -np.inf
    for seed in range(n_runs):
        try:
            labels = SpectralClustering(
                n_clusters=PART_CONFIG["C"]["n_clusters"],
                affinity="nearest_neighbors",
                n_neighbors=PART_CONFIG["C"]["n_neighbors"],
                assign_labels="kmeans",
                random_state=RANDOM_STATE + seed,
            ).fit_predict(Z)
        except Exception:
            continue
        if len(np.unique(labels)) < 2:
            continue
        if n_runs == 1:
            best_labels = labels
            break
        sample = min(2000, len(Z))
        score = silhouette_score(Z, labels, sample_size=sample,
                                  random_state=RANDOM_STATE)
        if score > best_score:
            best_score, best_labels = score, labels
    if best_labels is None:
        best_labels = SpectralClustering(
            n_clusters=PART_CONFIG["C"]["n_clusters"],
            affinity="nearest_neighbors",
            n_neighbors=PART_CONFIG["C"]["n_neighbors"],
            assign_labels="kmeans",
            random_state=RANDOM_STATE,
        ).fit_predict(Z)
    return relabel_by_cluster_size(best_labels), Z


def cluster_part_d(df):
    """Cluster Part D using Ward agglomerative clustering on a spatially augmented embedding.

    Concatenates the molecular PCA embedding with block-standardised spatial
    neighbourhood statistics (mean and std of neighbours' embeddings) and raw
    spatial features. Weights and K are taken from PART_CONFIG, which is set
    by the 3D sweep above.
    """
    cols = train_feature_choices["D"]["molecular_cols"]
    Z, _ = fit_numeric_embedding(df, cols, pca_dim=PART_CONFIG["D"]["pca_dim"])
    coords = df[["x_coord", "y_coord"]].to_numpy()
    # k-NN graph on spatial coordinates (k=8 neighbours per spot)
    graph = kneighbors_graph(coords, n_neighbors=PART_CONFIG["D"]["spatial_neighbors"],
                              mode="connectivity", include_self=False)
    row_sum = np.asarray(graph.sum(axis=1)).ravel()
    denom = np.maximum(row_sum[:, None], 1)
    # Neighbourhood mean and std capture spatial context for each spot
    neighbor_mean = (graph @ Z) / denom
    neighbor_sq   = (graph @ (Z ** 2)) / denom
    neighbor_std  = np.sqrt(np.maximum(neighbor_sq - neighbor_mean ** 2, 0.0))
    spatial_block = df[SPATIAL].to_numpy()

    if USE_BLOCK_STANDARDIZATION:
        # Each block is centred and divided by its global RMS so weight
        # multipliers have a consistent, scale-invariant interpretation.
        Z_blk  = standardize_block(Z)
        nm_blk = standardize_block(neighbor_mean)
        ns_blk = standardize_block(neighbor_std)
        sp_blk = standardize_block(spatial_block)
        sp_w   = PART_CONFIG["D"]["spatial_coord_weight"]
    else:
        Z_blk, nm_blk, ns_blk, sp_blk = Z, neighbor_mean, neighbor_std, spatial_block
        sp_w = 1.0

    embedding = np.hstack([
        Z_blk,
        PART_CONFIG["D"]["smooth_weight_mean"] * nm_blk,
        PART_CONFIG["D"]["smooth_weight_std"]  * ns_blk,
        sp_w * sp_blk,
    ])
    labels = AgglomerativeClustering(
        n_clusters=PART_CONFIG["D"]["n_clusters"],
        linkage="ward",
    ).fit_predict(embedding)
    return relabel_by_cluster_size(labels), embedding


## 5. Training-Set Metrics

Run all four pipelines on the training data and compute internal clustering
quality metrics (silhouette, Davies-Bouldin, Calinski-Harabasz). Since no
ground-truth labels are available, these metrics serve as a sanity check that
the pipelines produce non-degenerate clusterings.

In [ ]:
PART_FUNCS = {
    "A": cluster_part_a,
    "B": cluster_part_b,
    "C": cluster_part_c,
    "D": cluster_part_d,
}

ablation_rows = []
train_predictions = {}
for part, fn in PART_FUNCS.items():
    df_p = train[train["part"] == part].reset_index(drop=True)
    labels, Z = fn(df_p)
    train_predictions[part] = (labels, Z)
    m = clustering_metrics(Z, labels)
    m["part"] = part
    m["n_rows"] = len(df_p)
    ablation_rows.append(m)

abl = pd.DataFrame(ablation_rows)[
    ["part", "n_rows", "n_clusters", "silhouette",
     "davies_bouldin", "calinski_harabasz"]
]
print("Train-side metrics (v8 pipeline):")
print(abl.to_string(index=False))

Train-side metrics (v8 pipeline):
part  n_rows  n_clusters  silhouette  davies_bouldin  calinski_harabasz
   A    1400          10    0.393063        1.425049         331.981174
   B    1600           7    0.200810        1.941245         146.927184
   C    1800          16    0.096378        2.782150          81.475612
   D    1800          10    0.121634        2.665320         102.229040


In [ ]:
# Print cluster size distribution per part to verify no degenerate outputs
# (e.g., one cluster absorbing everything, or every point in its own cluster).
for part, (labels, _) in train_predictions.items():
    sizes = summarize_cluster_sizes(labels)
    print(f"\nPart {part}: {len(sizes)} clusters")
    print(f"  largest:  {sizes['size'].max()} ({100*sizes['size'].max()/sizes['size'].sum():.1f}%)")
    print(f"  smallest: {sizes['size'].min()}")
    print(f"  median:   {int(sizes['size'].median())}")


Part A: 10 clusters
  largest:  173 (12.4%)
  smallest: 62
  median:   148

Part B: 7 clusters
  largest:  380 (23.8%)
  smallest: 160
  median:   200

Part C: 16 clusters
  largest:  145 (8.1%)
  smallest: 80
  median:   111

Part D: 10 clusters
  largest:  281 (15.6%)
  smallest: 111
  median:   169


## 6. Final Submission on test.csv

In [ ]:
# Apply each part-specific pipeline to the corresponding test rows.
# Clustering is fitted directly on test data (unsupervised — no train labels used).
test_parts = {p: test[test["part"] == p].copy() for p in ["A", "B", "C", "D"]}
test_predictions = {}

for part, fn in PART_FUNCS.items():
    df_p = test_parts[part].reset_index(drop=True)
    labels, _ = fn(df_p)
    test_predictions[part] = pd.DataFrame({
        "row_id": df_p["row_id"].values,
        "cluster_id": labels,
    })
    print(f"Part {part}: {len(df_p)} rows -> {len(np.unique(labels))} clusters")

Part A: 1200 rows -> 10 clusters
Part B: 1400 rows -> 6 clusters
Part C: 1600 rows -> 16 clusters
Part D: 1600 rows -> 10 clusters


In [ ]:
# Concatenate all part predictions and re-align to the original test row order.
# Assertions guard against missing predictions and row-order mismatches.
submission = pd.concat(list(test_predictions.values()), ignore_index=True)
submission = (test[["row_id"]]
              .merge(submission, on="row_id", how="left"))

assert submission["cluster_id"].notna().all(), "Missing predictions!"
assert (submission["row_id"].values == test["row_id"].values).all(), "Row order mismatch!"
submission["cluster_id"] = submission["cluster_id"].astype(int)

out_path = f"{DATA_DIR}/submission_v12.csv"
submission.to_csv(out_path, index=False)
print(f"Wrote {out_path}")
print(submission.head())


In [ ]:
# Verify that the predicted number of clusters falls within the recommended
# range for each part. Out-of-range solutions may indicate a degenerate result.
RECOMMENDED_K = {"A": (5, 15), "B": (6, 20), "C": (8, 25), "D": (8, 30)}
for part, (lo, hi) in RECOMMENDED_K.items():
    pred = test_predictions[part]["cluster_id"]
    k = pred.nunique()
    flag = "OK" if lo <= k <= hi else "WARN: out of recommended range"
    print(f"Part {part}: k={k}  (recommended {lo}-{hi})  [{flag}]")

Part A: k=10  (recommended 5-15)  [OK]
Part B: k=6  (recommended 6-20)  [OK]
Part C: k=16  (recommended 8-25)  [OK]
Part D: k=10  (recommended 8-30)  [OK]


## 7. Notes

**Metric interpretation**
The training-set internal metrics (silhouette, Davies-Bouldin, Calinski-Harabasz) are sanity checks, not selection criteria. For Part C in particular, a higher silhouette does not imply a better clustering — the true cluster structure is manifold-like rather than Euclidean-compact, so bootstrap stability is the more reliable signal.

**Block standardisation (Part D)**
`smooth_weight_mean`, `smooth_weight_std`, and `spatial_coord_weight` are only interpretable after block standardisation is applied. Without it, the block with the largest natural scale dominates Euclidean distance regardless of the chosen weight values.

**Fallback behaviour**
If UMAP or HDBSCAN is unavailable at runtime, Part B automatically falls back to a DBSCAN epsilon sweep on the PCA embedding. If the Part D 3D sweep finds no stable configuration, the pipeline falls back to fixed defaults.
